# Census Data Cleaning: Iowa Incorporated-Place Population (2020-2025 Annual)

Cleans the U.S. Census Bureau **subcounty population estimates** for Iowa incorporated places (series `SUB-IP-EST2025-POP-19`) into a tidy table: one row per `(place, year, estimate_type)`.

**Input:**  `data/tabular/01_raw/census/Iowa-Census-2020-2025.xlsx`  
**Output:** `data/tabular/02_clean/census/iowa-census-population-2020-2025-clean.csv`

**What's in it** - every Iowa city, with the **April 1, 2020 estimates base** and the **July 1 estimates for 2020-2025**. (Unlike the 2010-2020 intercensal file, this annual series carries no decennial census column.) The raw `.xlsx` is a single sheet with a two-row header, a title banner on top, and citation/footnote prose at the bottom - none of which is data.

**Pipeline**
1. **Load** the sheet raw (no header).
2. **Locate** the place rows by content, discarding banner and footnotes.
3. **Map** each column to a `(year, estimate_type)` from the two header rows.
4. **Parse** `"<Name> city, Iowa"` into `place` + `place_type`.
5. **Reshape** wide -> long (tidy), one row per place/year/estimate_type.
6. **Enforce** a unique key and sanity-check the counts.
7. **Save**.

In [1]:
import re
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "census"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "census"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/census
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/census


## Step 1 - Load

Read the whole sheet with no header; the genuine header spans two rows (labels in row 2, years in row 3) which we interpret in Step 3.

In [2]:
RAW_FILE = "Iowa-Census-2020-2025.xlsx"
# header=None: the real header spans rows 2-3 and the sheet is wrapped in title
# rows above and citation/footnote rows below, so we parse positions ourselves.
raw = pd.read_excel(RAW_DIR / RAW_FILE, header=None)
print(f"Loaded {raw.shape[0]:,} rows x {raw.shape[1]} cols from {RAW_FILE}")
raw.head(6)

Loaded 949 rows x 8 cols from Iowa-Census-2020-2025.xlsx


,0,1,2,3,4,5,6,7
0,table with row headers in column A and column ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Annual Estimates of the Resident Population fo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Geographic Area,"April 1, 2020\nEstimates Base",Population Estimate (as of July 1),NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,2020,2021.0,2022.0,2023.0,2024.0,2025.0
4,"Ackley city, Iowa",1596,1591,1579.0,1560.0,1541.0,1535.0,1530.0
5,"Ackworth city, Iowa",117,117,120.0,119.0,119.0,121.0,121.0


## Step 2 - Locate the data block

Strip the title banner above and the citation/footnote prose below by keeping only genuine place rows, detected by content rather than by row index.

In [3]:
# Place rows are exactly those whose first cell ends in ", Iowa" and whose
# second cell (the April 1 base) is numeric. This cleanly excludes the title
# rows above and the citation/footnote rows below without hard-coding indices.
is_place = (
    raw[0].astype("string").str.endswith(", Iowa", na=False)
    & pd.to_numeric(raw[1], errors="coerce").notna()
)
block = raw[is_place].copy()
assert is_place.any(), "No place rows detected -- sheet layout changed?"
print(f"{len(block):,} place rows detected "
      f"(raw rows {block.index.min()}-{block.index.max()})")

939 place rows detected (raw rows 4-942)


## Step 3 - Map columns to (year, estimate_type)

The sheet packs three kinds of figure side by side: an **April 1 estimates base**, a run of **July 1 estimates** (one column per year), and (for this annual series, no decennial census column). We read the two header rows to assign each column a `year` and an `estimate_type`.

In [4]:
# The two header rows: row index 2 holds the column *labels*
# (e.g. "April 1, 2010\nEstimates Base", "April 1, 2020\nCensus") and row
# index 3 holds the *year* under each July-1 estimate column. We read both to
# map every data column -> (year, estimate_type) instead of hard-coding offsets.
labels = raw.iloc[2].astype("string").str.replace(r"\s+", " ", regex=True).fillna("")
year_row = raw.iloc[3]

colmap: dict[int, tuple[int, str]] = {}
for col in range(1, raw.shape[1]):
    yr = pd.to_numeric(year_row[col], errors="coerce")
    if pd.notna(yr):
        colmap[col] = (int(yr), "july_estimate")
        continue
    label = labels[col]
    m = re.search(r"(\d{4})", label)
    if not m:
        continue  # spacer / empty column
    yr = int(m.group(1))
    if "Estimates Base" in label:
        colmap[col] = (yr, "estimates_base")
    elif "Census" in label:
        colmap[col] = (yr, "census")
    else:
        raise ValueError(f"Unrecognized April-1 column label: {label!r}")

assert any(t == "estimates_base" for _, t in colmap.values()), "no estimates base col"
print("Column -> (year, estimate_type):")
for col, (yr, t) in sorted(colmap.items()):
    print(f"  col {col:>2}  {yr}  {t}")

Column -> (year, estimate_type):
  col  1  2020  estimates_base
  col  2  2020  july_estimate
  col  3  2021  july_estimate
  col  4  2022  july_estimate
  col  5  2023  july_estimate
  col  6  2024  july_estimate
  col  7  2025  july_estimate


## Step 4 - Parse place names

Split `"Ackley city, Iowa"` into `place` + `place_type`, dropping the redundant `, Iowa` (this series is Iowa-only).

In [5]:
# Every place is "<Name> city, Iowa" in this series; we still parse the type so
# the schema generalizes if towns/CDPs ever appear, and guard that all parse.
_PLACE_RE = re.compile(r"^(?P<place>.+) (?P<place_type>city|town|village|CDP), Iowa$")

parsed = block[0].astype("string").str.extract(_PLACE_RE)
unparsed = block.loc[parsed["place"].isna(), 0].tolist()
assert not unparsed, f"Unparseable place names: {unparsed[:5]}"

block["place"] = parsed["place"]
block["place_type"] = parsed["place_type"]
print(f"{block['place'].nunique():,} places; "
      f"types: {block['place_type'].value_counts().to_dict()}")

939 places; types: {'city': 939}


## Step 5 - Reshape wide to long

Each place row carries 7 population figures across the columns; we melt them into one row apiece. The April-1 estimates base is tagged with its own `estimate_type` so it is never conflated with the July-1 estimates (2020 therefore appears twice: as `estimates_base` and `july_estimate`).

In [6]:
# Wide -> long (tidy): one row per (place, year, estimate_type). The April-1
# base and (where present) the decennial census count are kept as their own
# estimate_type so they are never silently averaged with the July-1 estimates.
long = block.melt(
    id_vars=["place", "place_type"],
    value_vars=list(colmap),
    var_name="_col",
    value_name="population",
)
long["year"] = long["_col"].map(lambda c: colmap[c][0])
long["estimate_type"] = long["_col"].map(lambda c: colmap[c][1])

long["population"] = pd.to_numeric(long["population"], errors="coerce")
n_drop = long["population"].isna().sum()
long = long.dropna(subset=["population"])
long["population"] = long["population"].astype(int)

clean = (
    long[["place", "place_type", "year", "estimate_type", "population"]]
    .sort_values(["place", "year", "estimate_type"])
    .reset_index(drop=True)
)
print(f"Reshaped to {len(clean):,} long rows ({n_drop:,} blank cells dropped)")
clean.head()

Reshaped to 6,573 long rows (0 blank cells dropped)


,place,place_type,year,estimate_type,population
0,Ackley,city,2020,estimates_base,1596
1,Ackley,city,2020,july_estimate,1591
2,Ackley,city,2021,july_estimate,1579
3,Ackley,city,2022,july_estimate,1560
4,Ackley,city,2023,july_estimate,1541


## Step 6 - Enforce key & sanity check

In [7]:
KEY = ["place", "year", "estimate_type"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"

# Populations are counts: strictly positive, no absurd outliers for IA cities.
assert clean["population"].gt(0).all(), "non-positive population!"
assert clean["population"].max() < 1_000_000, "implausible city population!"

print(f"Key unique across {len(clean):,} rows.")
print(f"places: {clean['place'].nunique():,}  |  "
      f"years: {clean['year'].min()}-{clean['year'].max()}")
print(f"estimate_type counts: {clean['estimate_type'].value_counts().to_dict()}\n")
print("Largest places (latest july_estimate):")
latest = clean[clean.estimate_type == "july_estimate"]
latest = latest[latest.year == latest.year.max()]
print(latest.nlargest(5, "population")[["place", "year", "population"]].to_string(index=False))

Key unique across 6,573 rows.
places: 939  |  years: 2020-2025
estimate_type counts: {'july_estimate': 5634, 'estimates_base': 939}

Largest places (latest july_estimate):
       place  year  population
  Des Moines  2025      212086
Cedar Rapids  2025      137935
   Davenport  2025      100358
  Sioux City  2025       86356
      Ankeny  2025       77833


## Step 7 - Save

In [8]:
out_file = CLEAN_DIR / "iowa-census-population-2020-2025-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")

Saved 6,573 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/census/iowa-census-population-2020-2025-clean.csv
